# PySpark 6: structured streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-06-structured-streaming.ipynb)

Structured Streaming runs the same DataFrame logic on data that keeps arriving. This module shows a scheduled streaming job that reads new files, deduplicates within a watermark, and upserts each micro-batch, then shows Spark 4's `transformWithState` API for custom per-key state.

*Last reviewed: November 2026, against Apache Spark 4.3 (runnable cells pinned to PySpark 4.0).*

**You will learn to:**

- Read a directory of files as a stream
- Use a watermark and `dropDuplicatesWithinWatermark`
- Write each micro-batch with `foreachBatch`
- Run on a schedule with `trigger(availableNow=True)`

**Before you start:** PySpark 1 to 5.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

## How do you process streaming data with PySpark Structured Streaming?

Structured Streaming lets you express streaming logic with DataFrame APIs. Spark runs the query continuously or in micro-batches.

The cell below lands two JSON files in a directory (the second repeats an event), streams them
with a watermark and `dropDuplicatesWithinWatermark`, and writes each micro-batch to a Parquet
table with `foreachBatch`. Each batch overwrites its own `batch_id=` folder, so a batch that Spark
retries after a failure replaces its earlier attempt instead of adding duplicate rows. The batch stays a
distributed DataFrame: never `collect()` it to the driver inside `foreachBatch`. `trigger(availableNow=True)` processes everything available
and stops - the scheduled-batch pattern. On Databricks the source would be Auto Loader
(`format("cloudFiles")`) and the sink a Delta `MERGE`.

In [ ]:
landing_dir = os.path.join(WORK_DIR, "landing", "events")
checkpoint_dir = os.path.join(WORK_DIR, "checkpoints", "events_silver")
os.makedirs(landing_dir, exist_ok=True)
landed = [
    [("e1", "u1", "2026-01-06T09:00:00", "view"), ("e2", "u1", "2026-01-06T09:02:00", "add_to_cart")],
    [("e2", "u1", "2026-01-06T09:02:00", "add_to_cart"), ("e3", "u2", "2026-01-06T09:05:00", "purchase")],
]
columns = ["event_id", "user_id", "event_ts", "event_type"]
for index, batch in enumerate(landed):
    with open(os.path.join(landing_dir, f"part-{index}.json"), "w") as handle:
        handle.write("\n".join(json.dumps(dict(zip(columns, row))) for row in batch))

silver_dir = os.path.join(WORK_DIR, "silver", "events")
batch_ids = []


def merge_batch(batch_df, batch_id):
    batch_ids.append(batch_id)
    # Idempotent per batch: a retried batch overwrites its own folder instead of appending twice.
    batch_df.write.mode("overwrite").parquet(os.path.join(silver_dir, f"batch_id={batch_id}"))


def silver_event_ids():
    silver = spark.read.parquet(silver_dir)
    return sorted(row.event_id for row in silver.select("event_id").distinct().collect())  # tiny result


def run_stream():
    query = (
        spark.readStream
        .schema("event_id string, user_id string, event_ts timestamp, event_type string")
        .json(landing_dir)
        .withWatermark("event_ts", "30 minutes")
        .dropDuplicatesWithinWatermark(["event_id"])
        .writeStream
        .foreachBatch(merge_batch)
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()


run_stream()
print("silver events:", silver_event_ids(), "batches:", batch_ids)
assert silver_event_ids() == ["e1", "e2", "e3"]

batches_before = len(batch_ids)
run_stream()  # same checkpoint: the files are already committed, so nothing is reprocessed
assert len(batch_ids) == batches_before
print("second run read nothing new - the checkpoint remembers progress")

Real-world points:

+ Checkpoint locations are required for fault tolerance.
+ Watermarks define how late data can arrive before Spark drops state.
+ Deduplication needs a stable event key.
+ Streaming writes must be idempotent or carefully checkpointed.

How it works under the hood. A streaming query is a batch plan re-executed on every trigger. State for deduplication and aggregations lives in the state store under `checkpointLocation`, and a watermark bounds how long that state is kept. The driver keeps the query running and schedules one micro-batch job per trigger; the sink commits each batch transactionally.

Worked example. A 5-minute tumbling window with a 30-minute watermark keeps at most about six windows of state per key; drop the watermark and the state store grows for the life of the query.

Common mistake: `outputMode("complete")` with an unbounded key space, which recomputes and rewrites the whole result on every batch. Use `append` or `update` with a watermark for streaming sinks.

## How do you keep custom per-key state in a stream?

Built-in stateful operators (windowed aggregation, `dropDuplicatesWithinWatermark`, stream-stream joins) cover most pipelines. When the rule is your own - a session that closes after 30 idle minutes, a running balance, "alert on the third failed payment" - Spark 4 adds `transformWithState`, called `transformWithStateInPandas` in Python. It replaces `applyInPandasWithState` (and `mapGroupsWithState` / `flatMapGroupsWithState` in Scala) as the arbitrary-state API.

You write a `StatefulProcessor`. `init` declares named state variables - `ValueState`, `ListState` (cheap appends) or `MapState` (point lookups), each with an optional TTL that evicts expired state for you. `handleInputRows` runs once per key per micro-batch, and timers (processing or event time) fire `handleExpiredTimer`:

```python
from typing import Iterator

import pandas as pd
from pyspark.sql.streaming import StatefulProcessor, StatefulProcessorHandle

spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider",
)


class EventCounter(StatefulProcessor):
    def init(self, handle: StatefulProcessorHandle) -> None:
        self.seen = handle.getValueState("seen", "n long")  # one value per user_id

    def handleInputRows(self, key, rows, timerValues) -> Iterator[pd.DataFrame]:
        total = self.seen.get()[0] if self.seen.exists() else 0
        for batch in rows:  # pandas DataFrames holding this key's new rows
            total += len(batch)
        self.seen.update((total,))
        yield pd.DataFrame({"user_id": [key[0]], "events_seen": [total]})

    def close(self) -> None:
        pass


counts = (
    events_stream.groupBy("user_id")
    .transformWithStateInPandas(
        statefulProcessor=EventCounter(),
        outputStructType="user_id string, events_seen long",
        outputMode="Update",
        timeMode="None",  # "ProcessingTime" or "EventTime" to register timers
    )
)
```

The state lives in the checkpoint's state store, so the same checkpoint rules apply: one checkpoint per query, never shared. The PySpark API reference example sets the RocksDB state store provider as above, and the Python worker needs `pandas`, `pyarrow` and `protobuf` installed (the `pyspark[connect]` extra brings `protobuf`).

Common mistake: keeping per-key state with no TTL or timer. A key that never comes back holds its state forever, and the state store grows until checkpoints slow down. Give every state variable a TTL or clear it from a timer.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Checkpoints make a streaming query restartable. Keep one per query, and never share it.
- A watermark bounds how late data can arrive and how much state Spark keeps.
- On Databricks, use Auto Loader and a Delta MERGE instead of this local pattern.
- For custom per-key rules, use `transformWithStateInPandas` with a TTL or timer on every state variable.

**Next:** [PySpark 7: monitoring and performance](pyspark-07-monitoring-and-performance.ipynb)